# When a projection eats the leftover

Some corrections are projections: doing them twice is the same as doing them once. For those, applying the correction to the leftover gives zero. Doing leftover twice gives the same leftover.

This notebook uses the shipped calculator. Arithmetic is exact (fractions, not floating point).

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == "jupyter_notebooks" else cwd
code_dir = (root / "code").resolve()
assert (code_dir / "coda_a_core.py").is_file(), "Run from this package root or jupyter_notebooks/."
sys.path[:] = [str(code_dir)] + [entry for entry in sys.path if entry != str(code_dir)]
import coda_a_core as companion
assert Path(companion.__file__).resolve() == code_dir / "coda_a_core.py", "An ambient module shadowed the local companion."
import cochain_presentation as cochain
assert Path(cochain.__file__).resolve() == code_dir / "cochain_presentation.py", "An ambient solver shadowed the local solver."

from sympy import Matrix
from coda_a_core import is_projection, residual


## Doing it twice is the same as once

Keep-the-first-coordinate is a projection: $C(C(a,b))=C(a,0)=(a,0)=C(a,b)$. The leftover of $(3,4)$ is $(0,4)$. Feed that leftover back to $C$ and you get $(0,0)$. Feed it to leftover again and you still get $(0,4)$.

In [2]:
C = Matrix([[1, 0], [0, 0]])
x = Matrix([[3], [4]])
print("C twice equals C?", C * C == C)
print("is_projection?", is_projection(C))
r = residual(C, x)
print("leftover:", list(r))
print("C of leftover:", list(C * r))
print("leftover of leftover:", list(residual(C, r)))
assert is_projection(C) is True
assert (C * r).is_zero_matrix is True
assert residual(C, r) == r

C twice equals C? True
is_projection? True
leftover: [0, 4]
C of leftover: [0, 0]
leftover of leftover: [0, 4]


## Not every correction is a projection

Let $N(a,b)=(b,0)$. Then $N(N(a,b))=N(b,0)=(0,0)$, which is not $N$. For $(0,1)$ the leftover is $(-1,1)$, and $N$ does *not* eat it: $N(-1,1)=(1,0)$.

In [3]:
N = Matrix([[0, 1], [0, 0]])
y = Matrix([[0], [1]])
print("N twice equals N?", N * N == N)
print("is_projection?", is_projection(N))
s = residual(N, y)
print("leftover:", list(s))
print("N of leftover:", list(N * s))
print("leftover of leftover:", list(residual(N, s)))
assert is_projection(N) is False
assert s == Matrix([[-1], [1]])
assert (N * s).is_zero_matrix is False
assert residual(N, s) != s

N twice equals N? False
is_projection? False
leftover: [-1, 1]
N of leftover: [1, 0]
leftover of leftover: [-2, 1]
